In [ ]:
from getpass import getpass

admin_rdm_url = 'https://admin.bh.rdm.yzwlab.com/' #'https://admin.staging.rdm.example.com/'
rdm_url = 'https://bh.rdm.yzwlab.com/'

idp_name_integrated_admin = None
idp_username_integrated_admin = None
idp_password_integrated_admin = None

idp_name_quota_test_1 = None
idp_username_quota_test_1 = None
idp_password_quota_test_1 = None

# idp_username_quota_test_1 が所属する機関名
target_organization = None

# idp_username_quota_test_1 でログインした際にIdPから返されるメールアドレス（Adminのユーザ検索に使用）
email_search = None
default_result_path = None
close_on_fail = False
transition_timeout = 60000

In [ ]:
if idp_name_integrated_admin is None:
    idp_name_integrated_admin = input(prompt='IdP name for integrated_admin')
if idp_username_integrated_admin is None:
    idp_username_integrated_admin = input(prompt=f'Username for {idp_name_integrated_admin}')
if idp_password_integrated_admin is None:
    idp_password_integrated_admin = getpass(prompt=f'Password for {idp_username_integrated_admin}@{idp_name_integrated_admin}')
(len(idp_username_integrated_admin), len(idp_password_integrated_admin))

In [ ]:
if idp_name_quota_test_1 is None:
    idp_name_quota_test_1 = input(prompt='IdP name for quota_test_1')
if idp_username_quota_test_1 is None:
    idp_username_quota_test_1 = input(prompt=f'Username for {idp_name_quota_test_1}')
if idp_password_quota_test_1 is None:
    idp_password_quota_test_1 = getpass(prompt=f'Password for {idp_username_quota_test_1}@{idp_name_quota_test_1}')
if email_search is None:
    email_search = input(prompt='Email address of quota test user 1 (Admin ユーザ検索欄の入力値)')
if target_organization is None:
    target_organization = input(prompt='Target organization name for quota test (対象機関名)')
(len(idp_username_quota_test_1), len(idp_password_quota_test_1))

In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

# 再ログイン_NII

- サブシステム名: 再ログインサイクル -> delete
- ページ/アドオン: ログイン
- 機能分類: 再ログインサイクル -> delete
- シナリオ名: 再ログインサイクル -> delete
- 用意するテストデータ: URL一覧、アカウント(クォータテストユーザー1), アカウント(統合管理者)

In [ ]:
import importlib
import pandas as pd

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path)

## ウェブブラウザの同一ウィンドウでGRDMトップページを表示する

GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await page.goto(rdm_url)

    # 同意する ボタンが現れるまで待つ
    await expect(page.locator('//button[text() = "同意する"]')).to_be_visible(timeout=transition_timeout)

    # 同意する をクリック
    await page.locator('//button[text() = "同意する"]').click()

    # 同意する が表示されなくなったことを確認
    await expect(page.locator('//button[text() = "同意する"]')).to_have_count(0, timeout=500)

await run_pw(_step)

## 「RCOS IdP」を利用し、クォータテストユーザー1としてログインする

設定画面が表示されること

In [ ]:
async def _step(page):
    await scripts.grdm.login(
        page, idp_name_quota_test_1, idp_username_quota_test_1, idp_password_quota_test_1, transition_timeout=transition_timeout
    )

    await expect(page.locator('h2.page-header')).to_contain_text('設定', timeout=transition_timeout)

await run_pw(_step)

※ (補足)ここでは新規ユーザーのUserQuotaに関する手順を実施します

※ (補足)新規ユーザーは初回ログイン時点でプロフィールの必須項目（「姓」「名前」「姓（英語）」「名前（英語）」等）が未入力であるため、ダッシュボードではなく「設定」画面へ遷移する。これは意図した挙動である。

## ウェブブラウザの同一ウィンドウでGakunin RDM管理者のトップページを表示する

管理者トップページが表示されること

In [ ]:
async def _step(page):
    await page.goto(admin_rdm_url)

    await expect(page.locator('.login-logo')).to_be_visible(timeout=30000)

await run_pw(_step)

## ログイン情報を用いてGakuNin RDMにログインする

個人の管理者ページが表示されること

In [ ]:
async def _step(page):
    await scripts.grdm.login_as_admin(
        page, idp_name_integrated_admin, idp_username_integrated_admin, idp_password_integrated_admin, transition_timeout=transition_timeout
    )

    await expect(page.locator('//*[contains(@class, "btn-danger") and contains(text(), "ログアウト")]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## 「NIIストレージのクォータ」を選択する

「NIIストレージ使用の機関のリスト」が表示されること

In [ ]:
async def _step(page):
    await page.locator('//a[@href = "/institutions/institution_list/"]').click()

    await expect(page.locator('//h2[text() = "NIIストレージ使用の機関のリスト"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「機関のリスト」画面の名前から対象機関を選択する

「NIIストレージの統計ステータス」画面が表示されること

In [ ]:
import traceback

async def _step(page):
    while True:
        link = page.locator(f'//a[normalize-space() = "{target_organization}"]')
        try:
            await expect(link).to_be_visible()
        except:
            traceback.print_exc()
            print('Search next page...')
            # 次のページかもしれない
            await page.locator('//a[i[contains(@class, "fa-angle-right")]]').click()
            await expect(page.locator('//h2[text() = "NIIストレージ使用の機関のリスト"]')).to_be_visible(timeout=transition_timeout)
            continue
        await link.click()
        break

    await expect(page.locator('//h2[contains(text(), "NIIストレージの統計ステータス")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 検索フォームのGUID欄またはeメール欄に新規作成されたクォータテストユーザー1の情報を入力し、「検索」ボタンを押下する

該当のユーザーが一覧に表示され、「クォータ」列が100 GB（システム既定値）であることを確認する

In [ ]:
async def _step(page):
    await page.locator('#id_email').fill(email_search)
    await page.locator('//button[@type = "submit" and text() = "検索"]').click()

    row = page.locator(f'//tr[td[contains(text(), "{email_search}")]]')
    await expect(row).to_be_visible(timeout=transition_timeout)
    await expect(row.locator('td').last).to_contain_text('100 GB', timeout=transition_timeout)

await run_pw(_step)

## 「ユーザ管理」から「ユーザ管理」選択し、 ページを表示する

ユーザ検索画面が表示される

In [ ]:
async def _step(page):
    await page.locator('//a[@href = "#collapseUsers"]').click()
    await page.locator('//a[@href = "/users/"]').click()

    await expect(page.locator('//input[@name = "guid"]').first).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「ユーザ検索」画面のeメール欄へメールアドレスを入力し、「検索」ボタンを押下する

該当のユーザーが表示される

In [ ]:
async def _step(page):
    await page.locator('//input[@name = "email"]').fill(email_search)
    await page.locator('//input[@type = "submit"]').click()

    await expect(page.locator(f'//td[contains(text(), "{email_search}")]').first).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「ユーザ詳細」画面で「NIIストレージの割当て(GB)」欄の値を確認する

値が100であることを確認する

In [ ]:
async def _step(page):
    await expect(page.locator('#storageLimit')).to_have_value('100', timeout=transition_timeout)

await run_pw(_step)

## 「ユーザ詳細」画面で「GDPRアカウントの削除」ボタンをクリックする

「このユーザーをGDPR削除してもよろしいですか？」ダイアログが表示されること

In [ ]:
async def _step(page):
    await page.get_by_role("link", name="GDPRアカウントの削除").click()
    await expect( page.locator("#deleteModal h3")).to_contain_text("このユーザーをGDPR削除してもよろしいですか？")

await run_pw(_step)

## 「確認」ボタンをクリックする

「User <uid> was successfully GDPR deleted」メッセージが表示されること

In [ ]:
async def _step(page):
    uid = page.url.rstrip("/").split("/")[-1]
    print(uid)

    await page.locator('#deleteModal input[type="submit"][value="確認"]').click()
    await expect(page.get_by_role("alert").first).to_contain_text(f"User {uid} was successfully GDPR deleted")

await run_pw(_step)

## 「ユーザ管理」から「ユーザ管理」選択し、 ページを表示する

ユーザ検索画面が表示される

In [ ]:
async def _step(page):
    await page.locator('//a[@href = "#collapseUsers"]').click()
    await page.get_by_role("link", name="ユーザ管理").click()

    await expect(page.locator('//input[@name = "guid"]').first).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「ユーザ検索」画面のeメール欄へメールアドレスを入力し、「検索」ボタンを押下する

「User with email address <email_test> not found.」が表示されること

In [ ]:
async def _step(page):
    await page.locator('//input[@name = "email"]').fill(email_search)
    await page.locator('//input[@type = "submit"]').click()

    await expect(page.get_by_text(f'User with email address {email_search} not found.')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

終了処理を実施。

In [ ]:
await finish_pw_context()

In [ ]:
!rm -fr {work_dir}